In [2]:
spark.stop()

26/03/20 18:12:39 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-bac8fad1-c57c-4dbb-8448-bc5e51484704/pyspark-0da6e760-b5b6-4d4d-bf57-1eb59b89a020. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-bac8fad1-c57c-4dbb-8448-bc5e51484704/pyspark-0da6e760-b5b6-4d4d-bf57-1eb59b89a020
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:174)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:109)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:90)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1126)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4(ShutdownHookManager.scala:65)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun

In [1]:
from pyspark.sql import SparkSession

# spark = (
#     SparkSession.builder
#     .appName("Przygotowanie")
#     # .master("local[*]")
#     .master("spark://spark-master:7077")
#     .getOrCreate()
# )


spark = (
    SparkSession.builder
    .appName("Przygotowanie")
    .master("spark://spark-master:7077")
    # .master("local[*]")
    .config("spark.sql.warehouse.dir", "/opt/spark/warehouse/")
    .enableHiveSupport()
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/20 18:13:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
! ls /opt/spark/warehouse/

test_file	uam_categories.snappy.parquet  uam_orders.snappy.parquet
uam_categories	uam_offers.snappy.parquet


In [2]:
spark.sql("show tables").show()

26/03/20 18:13:31 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/03/20 18:13:31 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/03/20 18:13:33 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/03/20 18:13:33 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore UNKNOWN@172.18.0.5
26/03/20 18:13:33 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


+---------+----------+-----------+
|namespace| tableName|isTemporary|
+---------+----------+-----------+
|  default|uam_offers|      false|
+---------+----------+-----------+



In [4]:
!ls /opt/spark/warehouse/

test_file	uam_categories.snappy.parquet  uam_orders.snappy.parquet
uam_categories	uam_offers.snappy.parquet


In [5]:
spark.sql("drop table if exists uam_offers")
spark.sql("create table uam_offers as select * from `parquet`.`/opt/spark/warehouse/uam_offers.snappy.parquet`").show()

26/03/20 18:11:31 WARN ObjectStore: Failed to get database parquet, returning NoSuchObjectException
26/03/20 18:11:32 WARN ResolveSessionCatalog: A Hive serde table will be created as there is no table provider specified. You can set spark.sql.legacy.createHiveTableByDefault to false so that native data source table will be created instead.
26/03/20 18:11:32 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/03/20 18:11:32 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/03/20 18:11:32 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/03/20 18:11:32 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/03/20 18:11:32 WARN HiveMetaStore: Location: file:/opt/spark/warehouse/uam_offers specified for non-external table:uam_offers
chgrp: changing ownership of 'file:///opt/spark/warehouse/uam_offers': chow

++
||
++
++



In [3]:
!ls /home/jovyan/data/

ls: cannot access '/home/jovyan/data/': No such file or directory


In [3]:
spark.sql("select * from uam_offers").show()

26/03/20 18:13:39 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
[Stage 0:>                                                          (0 + 1) / 1]

+--------------------+--------------------+--------------------+-------------------+-------------------+--------+---------+------------+---------------------+----------------------+----------------------+----------------------+-------------+--------------------+---+
|            offer_id|          offer_name|           seller_id|      starting_time|        ending_time|duration|    types|buynow_price|auction_minimal_price|auction_starting_price|stock_initial_quantity|stock_current_quantity|category_leaf|          attributes| pv|
+--------------------+--------------------+--------------------+-------------------+-------------------+--------+---------+------------+---------------------+----------------------+----------------------+----------------------+-------------+--------------------+---+
|004172c9c5049a158...|Części samochodow...|eb4dd0d58f2b23c39...|2017-03-06 14:56:46|               NULL|    NULL|[BUY_NOW]|         170|                 NULL|                  NULL|                  

In [5]:
import shutil
import urllib.request


# catalog = "workspace"
schema = "default"
volume = "uam"

config = [
    {   
        'table': 'uam_categories',
        'url': 'https://drive.google.com/u/1/uc?id=1WLPzOczLl0ATgk2qyRpLUdrG6duDhkL2&export=download',
        'expected_row_cnt': 174
    },
    {
        'table': 'uam_orders',
        'url': 'https://drive.google.com/u/1/uc?id=1zt-YtImHn_48pgNWNQ-xZQSbcoxbrdHe&export=download',
        'expected_row_cnt': 100
    },
    {
        'table': 'uam_offers',
        'url': 'https://drive.google.com/u/1/uc?id=1tMBkjT_RQC4hPl0hCj9reNUjeM4wiIci&export=download',
        'expected_row_cnt': 188
    },
]


def download(source_url, local_temp_loc):
    urllib.request.urlretrieve(source_url,local_temp_loc)


def drop_table_if_exists(table):
    spark.sql(f"drop table if exists {schema}.{table}")


def create_table(table, local_temp_loc):
    spark.read.parquet(local_temp_loc).write.saveAsTable(table)


def validate(table, expected_cnt):
    assert spark.table(table).count() == expected_cnt


def load_data():
    for cfg in config:
        table = cfg['table']
        expected_row_cnt = cfg['expected_row_cnt']
        url = cfg['url']
        local_temp = f'/opt/spark/warehouse/{table}.snappy.parquet'

        # download(url, local_temp)
        drop_table_if_exists(table)
        create_table(table, f'file:/opt/spark/warehouse/{table}.snappy.parquet')
        validate(table, expected_row_cnt)


load_data()


26/03/20 17:52:17 WARN TaskSetManager: Lost task 0.0 in stage 1.0 (TID 1) (172.18.0.3 executor 1): java.io.IOException: Mkdirs failed to create file:/opt/spark/warehouse/uam_categories/_temporary/0/_temporary/attempt_202603201752162699770168924860428_0001_m_000000_1 (exists=false, cwd=file:/opt/spark/work/app-20260320175145-0000/1)
	at org.apache.hadoop.fs.ChecksumFileSystem.create(ChecksumFileSystem.java:515)
	at org.apache.hadoop.fs.ChecksumFileSystem.create(ChecksumFileSystem.java:500)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1195)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1175)
	at org.apache.parquet.hadoop.util.HadoopOutputFile.create(HadoopOutputFile.java:74)
	at org.apache.parquet.hadoop.ParquetFileWriter.<init>(ParquetFileWriter.java:347)
	at org.apache.parquet.hadoop.ParquetFileWriter.<init>(ParquetFileWriter.java:314)
	at org.apache.parquet.hadoop.ParquetOutputFormat.getRecordWriter(ParquetOutputFormat.java:484)
	at org.apache.parquet.hadoop

Py4JJavaError: An error occurred while calling o41.saveAsTable.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 1.0 failed 4 times, most recent failure: Lost task 0.3 in stage 1.0 (TID 4) (172.18.0.4 executor 0): java.io.IOException: Mkdirs failed to create file:/opt/spark/warehouse/uam_categories/_temporary/0/_temporary/attempt_202603201752162699770168924860428_0001_m_000000_4 (exists=false, cwd=file:/opt/spark/work/app-20260320175145-0000/0)
	at org.apache.hadoop.fs.ChecksumFileSystem.create(ChecksumFileSystem.java:515)
	at org.apache.hadoop.fs.ChecksumFileSystem.create(ChecksumFileSystem.java:500)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1195)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1175)
	at org.apache.parquet.hadoop.util.HadoopOutputFile.create(HadoopOutputFile.java:74)
	at org.apache.parquet.hadoop.ParquetFileWriter.<init>(ParquetFileWriter.java:347)
	at org.apache.parquet.hadoop.ParquetFileWriter.<init>(Pa

In [6]:
!ls /home/jovyan/data/

ls: cannot access '/home/jovyan/data/': No such file or directory


In [7]:
!ls /opt/spark/warehouse/

uam_categories		       uam_offers.snappy.parquet
uam_categories.snappy.parquet  uam_orders.snappy.parquet
